In [2]:
# ============================================
# MEMBER 3: ADVANCED MODELS & SMOTE
# Bank Telemarketing Prediction
# USING ORIGINAL FEATURES ONLY (15 COLUMNS)
# ============================================
!pip install xgboost 
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("MEMBER 3: ADVANCED MODELS (ORIGINAL 15 FEATURES ONLY)")
print("=" * 60)

# ============================================
# 1. LOAD ORIGINAL DATA (Strictly 15 Features)
# ============================================

# Load the cleaned data
df = pd.read_csv('data/processed/bankclean.csv')

# Original 15 features only (excluding engineered features to prevent data leakage)
original_features = ['age', 'job', 'marital', 'education', 'default', 
                     'balance', 'housing', 'loan', 'contact', 'month', 
                     'duration', 'campaign', 'pdays', 'previous', 'poutcome']

# Keep only original features and the target 'y'
df_original = df[original_features + ['y']].copy()

# Convert target to binary (1 = yes, 0 = no)
y = (df_original['y'] == 'yes').astype(int)
X = df_original.drop('y', axis=1)

print("✅ Data loaded (Strictly 15 Original Features)")
print(f"Total samples: {X.shape}")
print(f"Target: yes={sum(y)}, no={len(y)-sum(y)}")

# ============================================
# 2. CREATE TRAIN/TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================
# 3. AUTOMATICALLY CREATE ALL 4 SCENARIOS
# ============================================

# S1: All features (with duration)
X_train_S1 = X_train.copy()
X_test_S1 = X_test.copy()

# S2: Realistic (No duration)
X_train_S2 = X_train.drop(columns=['duration'])
X_test_S2 = X_test.drop(columns=['duration'])

# S3: Demographics only
demo_cols = ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan']
X_train_S3 = X_train[demo_cols]
X_test_S3 = X_test[demo_cols]

# S4: Previous campaign only
prev_cols = ['pdays', 'previous', 'poutcome']
X_train_S4 = X_train[prev_cols]
X_test_S4 = X_test[prev_cols]

print("\n" + "=" * 60)
print("4 SCENARIOS READY (Generated Automatically)")
print("=" * 60)
print(f"S1 (All features with duration): {X_train_S1.shape[1]} columns")
print(f"S2 (No duration - realistic): {X_train_S2.shape[1]} columns")
print(f"S3 (Demographics only): {X_train_S3.shape[1]} columns")
print(f"S4 (Previous campaign only): {X_train_S4.shape[1]} columns")

# ============================================
# 4. PREPROCESSING & EVALUATION FUNCTIONS
# ============================================

def create_preprocessor(X):
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    
    return ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

def evaluate_model(X_train, X_test, y_train, y_test, model, model_name, scenario_name, use_smote=False):
    preprocessor = create_preprocessor(X_train)
    
    if use_smote:
        pipeline = ImbPipeline([
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=42)),
            ('classifier', model)
        ])
    else:
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    # Get probabilities for ROC-AUC
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    return {
        'Model': model_name,
        'Scenario': scenario_name,
        'SMOTE': 'Yes' if use_smote else 'No',
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }

# ============================================
# 5. DEFINE ADVANCED MODELS (Tuned for performance)
# ============================================
# Note: SVM max_iter is set to 2000 so it doesn't run forever on this large dataset.
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, max_iter=2000, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, eval_metric='logloss')
}

scenarios = {
    'S1_All_WithDuration': (X_train_S1, X_test_S1),
    'S2_Realistic_NoDuration': (X_train_S2, X_test_S2),
    'S3_DemographicsOnly': (X_train_S3, X_test_S3),
    'S4_PreviousOnly': (X_train_S4, X_test_S4)
}

# ============================================
# 6. RUN ALL MODELS (Takes time - please wait!)
# ============================================

print("\n" + "=" * 60)
print("RUNNING ADVANCED MODELS (4 models × 4 scenarios × 2 SMOTE = 32 runs)")
print("Please wait, SVM and Random Forest will take a few minutes...")
print("=" * 60)

results = []

for scenario_name, (X_train_sc, X_test_sc) in scenarios.items():
    print(f"\n[{scenario_name}]")
    
    for model_name, model in models.items():
        print(f"  Training {model_name}...")
        
        # 1. Run Without SMOTE
        result_no = evaluate_model(X_train_sc, X_test_sc, y_train, y_test, model, model_name, scenario_name, use_smote=False)
        results.append(result_no)
        
        # 2. Run With SMOTE
        result_smote = evaluate_model(X_train_sc, X_test_sc, y_train, y_test, model, model_name, scenario_name, use_smote=True)
        results.append(result_smote)

# ============================================
# 7. PROCESS & SAVE THE 8 CSV FILES FOR MEMBER 4
# ============================================

results_df = pd.DataFrame(results)

# Create results folder if it doesn't exist
os.makedirs('results/tables', exist_ok=True)

# Loop to separate and save the 8 required files
print("\n" + "=" * 60)
print("SAVING 8 EXPORT FILES FOR MEMBER 4")
print("=" * 60)

for scenario_name in scenarios.keys():
    # Get just the prefix (e.g., "S1", "S2")
    prefix = scenario_name.split('_')[0]
    
    # Filter for Without SMOTE
    df_no = results_df[(results_df['Scenario'] == scenario_name) & (results_df['SMOTE'] == 'No')]
    file_no = f'results/tables/advanced_results_{prefix}.csv'
    df_no.to_csv(file_no, index=False)
    print(f"✓ Saved: {file_no}")
    
    # Filter for With SMOTE
    df_yes = results_df[(results_df['Scenario'] == scenario_name) & (results_df['SMOTE'] == 'Yes')]
    file_yes = f'results/tables/advanced_smote_results_{prefix}.csv'
    df_yes.to_csv(file_yes, index=False)
    print(f"✓ Saved: {file_yes}")

print("\n" + "=" * 60)
print("MEMBER 3 - ADVANCED MODELS COMPLETE! ALL FILES READY.")
print("=" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 4.0 MB/s  0:00:01m 2.8 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
MEMBER 3: ADVANCED MODELS (ORIGINAL 15 FEATURES ONLY)
✅ Data loaded (Strictly 15 Original Features)
Total samples: (45211, 15)
Target: yes=5289, no=39922

4 SCENARIOS READY (Generated Automatically)
S1 (All features with duration): 15 columns
S2 (No duration - realistic): 14 columns
S3 (Demographics only): 8 columns
S4 (Previous campaign only): 3 columns

RUNNING ADVANCED MODELS (4 models × 4 scenarios × 2 SMOTE = 32 runs)
Please wait, SVM and Random Forest will take a few minutes...

[S1_All_WithDuration]
  Training KNN...
  Training Random Forest...
  Training SVM (RBF)...
  Training XGBoost...

[S2_Realistic_NoDuration]
  Training KNN...
  Training Random Forest...
  Training SVM (RBF)...
  Training XGBoost...

[S3_DemographicsOnly]
  Training KNN...
  Training Random Forest...
  

In [3]:
import pandas as pd

# Load the S2 (Realistic Scenario) files to check the stats
df_no_smote = pd.read_csv('results/tables/advanced_results_S2.csv')
df_yes_smote = pd.read_csv('results/tables/advanced_smote_results_S2.csv')

print("--- S2 (Realistic) WITHOUT SMOTE ---")
display(df_no_smote)

print("\n--- S2 (Realistic) WITH SMOTE ---")
display(df_yes_smote)

--- S2 (Realistic) WITHOUT SMOTE ---


,Model,Scenario,SMOTE,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN,S2_Realistic_NoDuration,No,0.886542,0.538462,0.211720,0.303935,0.700038
1,Random Forest,S2_Realistic_NoDuration,No,0.894283,0.709016,0.163516,0.265745,0.791222
2,SVM (RBF),S2_Realistic_NoDuration,No,0.619485,0.128005,0.387524,0.192443,0.532974
3,XGBoost,S2_Realistic_NoDuration,No,0.896495,0.669444,0.227788,0.339915,0.793980



--- S2 (Realistic) WITH SMOTE ---


,Model,Scenario,SMOTE,Accuracy,Precision,Recall,F1,ROC-AUC
0,KNN,S2_Realistic_NoDuration,Yes,0.736702,0.238847,0.571834,0.336953,0.703521
1,Random Forest,S2_Realistic_NoDuration,Yes,0.840208,0.372948,0.536862,0.440139,0.779444
2,SVM (RBF),S2_Realistic_NoDuration,Yes,0.224151,0.107199,0.768431,0.188151,0.438709
3,XGBoost,S2_Realistic_NoDuration,Yes,0.887537,0.527665,0.369565,0.434686,0.771157
